In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy.interpolate import interp1d

In [ ]:
#### INPUT filepaths under appropriate sample name, can input as many cells / samples as you'd like####
sample_files = {
    "sample-1": [
        r,
    ],
    "sample-2": [
        r,
    ],
    "sample-3": [
        r,
    ],
}

#### plotting setup ####
fig, axes = plt.subplots(2, 1, figsize=(15, 12), sharex=True)
colors = plt.cm.tab20.colors  # Up to 20 unique colors

summary_data = []  # to store max vals and their corresponding voltages

#### process each sample ####
for i, (sample_name, file_list) in enumerate(sample_files.items()):
    charge_color = colors[(2 * i) % len(colors)]          # unique color for charge
    discharge_color = colors[(2 * i + 1) % len(colors)]   # unique color for discharge

    charge_valid_voltage_ranges = []   # reset for each sample
    charge_interp_diff = []            # reset for each sample

    discharge_valid_voltage_ranges = []  # reset for each sample
    discharge_interp_diff = []           # reset for each sample

    for file_path in file_list:
        df = pd.read_excel(file_path, sheet_name=1)
        charge_D_Li = []      # store charge diff coef data
        discharge_D_Li = []   # store discharge diff coef data
        file_label = file_path.split('\\')[-1][:-5]  # label for plotting

        ICI_steps = list(range(10, df['Cycle Index'].max() - 3))

        for step in ICI_steps:
            #### Charge analysis ####
            charge_data = df[(df['Cycle Index'] == step) & (df['Step Index'] == charge_step)].copy()
            charge_pause_data = df[(df['Cycle Index'] == step) & (df['Step Index'] == charge_pause_step)].copy()
            next_charge_pause_data = df[(df['Cycle Index'] == step + 1) & (df['Step Index'] == charge_pause_step)].copy()

            if not (charge_data.empty or charge_pause_data.empty or next_charge_pause_data.empty):
                try:
                    tau = charge_data['Test Time (s)'].iloc[-1] - charge_data['Test Time (s)'].iloc[0]  # charge duration
                    t0 = charge_pause_data['Test Time (s)'].iloc[0]
                    mask = (charge_pause_data['Test Time (s)'] - t0 >= 0.1) & (charge_pause_data['Test Time (s)'] - t0 <= 5)  # early pause data
                    dEt_data = charge_pause_data[mask]

                    if not dEt_data.empty:
                        sqrt_t = np.sqrt(dEt_data['Test Time (s)'] - t0)
                        voltage = dEt_data['Voltage (V)']
                        slope, _, _, _, _ = linregress(sqrt_t, voltage)
                        dEt = abs(slope)  # slope of E with sqrt time - linear
                        dEs = abs(next_charge_pause_data['Voltage (V)'].iloc[0] - charge_pause_data['Voltage (V)'].iloc[0])
                        Eeq_vals = charge_pause_data['Voltage (V)'].iloc[0]

                        if dEt != 0 and tau != 0:
                            D = (4 / (np.pi * 9)) * ((rp / tau) ** 2) * ((dEs / dEt) ** 2)
                            charge_D_Li.append({'Step': step, 'Diff_coef (cm^2/s)': D, 'Eeq (V)': Eeq_vals})
                except Exception as e:
                    print(f"{file_label} CHARGE: Error at step {step} - {e}")

            #### Discharge analysis ####
            discharge_data = df[(df['Cycle Index'] == step) & (df['Step Index'] == discharge_step)].copy()
            discharge_pause_data = df[(df['Cycle Index'] == step) & (df['Step Index'] == discharge_pause_step)].copy()
            next_discharge_pause_data = df[(df['Cycle Index'] == step + 1) & (df['Step Index'] == discharge_pause_step)].copy()

            if not (discharge_data.empty or discharge_pause_data.empty or next_discharge_pause_data.empty):
                try:
                    tau = discharge_data['Test Time (s)'].iloc[-1] - discharge_data['Test Time (s)'].iloc[0]  # discharge duration
                    t0 = discharge_pause_data['Test Time (s)'].iloc[0]
                    mask = (discharge_pause_data['Test Time (s)'] - t0 >= 0.1) & (discharge_pause_data['Test Time (s)'] - t0 <= 5)  # early pause data
                    dEt_data = discharge_pause_data[mask]

                    if not dEt_data.empty:
                        sqrt_t = np.sqrt(dEt_data['Test Time (s)'] - t0)
                        voltage = dEt_data['Voltage (V)']
                        slope, _, _, _, _ = linregress(sqrt_t, voltage)
                        dEt = abs(slope)  # slope of E with sqrt time - linear
                        dEs = abs(next_discharge_pause_data['Voltage (V)'].iloc[0] - discharge_pause_data['Voltage (V)'].iloc[0])
                        Eeq_vals = discharge_pause_data['Voltage (V)'].iloc[0]

                        if dEt != 0 and tau != 0:
                            D = (4 / (np.pi * 9)) * ((rp / tau) ** 2) * ((dEs / dEt) ** 2)
                            discharge_D_Li.append({'Step': step, 'Diff_coef (cm^2/s)': D, 'Eeq (V)': Eeq_vals})
                except Exception as e:
                    print(f"{file_label} DISCHARGE: Error at step {step} - {e}")

        charge_results_df = pd.DataFrame(charge_D_Li)
        discharge_results_df = pd.DataFrame(discharge_D_Li)

        if not charge_results_df.empty:
            axes[0].scatter(charge_results_df['Eeq (V)'], charge_results_df['Diff_coef (cm^2/s)'],
                            label=f"{sample_name} Charge", color=charge_color, s=30, edgecolors='black', alpha=0.6)

            charge_data_sorted = charge_results_df.sort_values('Eeq (V)')
            vmin = charge_data_sorted['Eeq (V)'].min()
            vmax = charge_data_sorted['Eeq (V)'].max()
            charge_valid_voltage_ranges.append((vmin, vmax))

            charge_interp_diff.append((charge_data_sorted['Eeq (V)'], charge_data_sorted['Diff_coef (cm^2/s)']))

        if not discharge_results_df.empty:
            axes[0].scatter(discharge_results_df['Eeq (V)'], discharge_results_df['Diff_coef (cm^2/s)'],
                            label=f"{sample_name} Discharge", color=discharge_color, s=30, edgecolors='black', alpha=0.6)

            discharge_data_sorted = discharge_results_df.sort_values('Eeq (V)')
            vmin = discharge_data_sorted['Eeq (V)'].min()
            vmax = discharge_data_sorted['Eeq (V)'].max()
            discharge_valid_voltage_ranges.append((vmin, vmax))

            discharge_interp_diff.append((discharge_data_sorted['Eeq (V)'], discharge_data_sorted['Diff_coef (cm^2/s)']))

    #### charge voltage grid for interpolation ####
    if charge_valid_voltage_ranges:
        vmin_all = max(v[0] for v in charge_valid_voltage_ranges)
        vmax_all = min(v[1] for v in charge_valid_voltage_ranges)
        voltage_grid = np.linspace(vmin_all, vmax_all, 1000)

        #### interpolate charge diffusion values ####
        interp_matrix = []
        for v_vals, d_vals in charge_interp_diff:
            interp_func = interp1d(v_vals, d_vals, kind='linear', bounds_error=False, fill_value=np.nan)
            interp_matrix.append(interp_func(voltage_grid))

        matrix = np.array(interp_matrix)

        #### mean & std dev ####
        mean_diff = np.nanmean(matrix, axis=0)
        std_diff = np.nanstd(matrix, axis=0)

        #### plot charge results ####
        axes[1].plot(voltage_grid, mean_diff, color=charge_color, linewidth=2.5, label=f"{sample_name} Charge Mean")
        axes[1].fill_between(voltage_grid,
                             mean_diff - std_diff,
                             mean_diff + std_diff,
                             color=charge_color, alpha=0.3)

        #### get max charge diffusion coefficient ####
        max_idx = np.nanargmax(mean_diff)
        max_voltage = voltage_grid[max_idx]
        max_diff_val = mean_diff[max_idx]
        max_std_val = std_diff[max_idx]

        summary_data.append({
            "Sample": sample_name,
            "Mode": "Charge",
            "Voltage (V)": round(max_voltage, 3),
            "Mean Diff Coef (cm^2/s)": max_diff_val,
            "Std Dev": max_std_val,
            "Diff Coef (cm^2/s)": f"{max_diff_val:.2e} ± {max_std_val:.2e}"
        })

    #### discharge voltage grid for interpolation ####
    if discharge_valid_voltage_ranges:
        vmin_all = max(v[0] for v in discharge_valid_voltage_ranges)
        vmax_all = min(v[1] for v in discharge_valid_voltage_ranges)
        voltage_grid = np.linspace(vmin_all, vmax_all, 1000)

        #### interpolate discharge diffusion values ####
        interp_matrix = []
        for v_vals, d_vals in discharge_interp_diff:
            interp_func = interp1d(v_vals, d_vals, kind='linear', bounds_error=False, fill_value=np.nan)
            interp_matrix.append(interp_func(voltage_grid))

        matrix = np.array(interp_matrix)

        #### mean & std dev ####
        mean_diff = np.nanmean(matrix, axis=0)
        std_diff = np.nanstd(matrix, axis=0)

        #### plot discharge results ####
        axes[1].plot(voltage_grid, mean_diff, color=discharge_color, linewidth=2.5, label=f"{sample_name} Discharge Mean")
        axes[1].fill_between(voltage_grid,
                             mean_diff - std_diff,
                             mean_diff + std_diff,
                             color=discharge_color, alpha=0.3)

        #### get max discharge diffusion coefficient ####
        max_idx = np.nanargmax(mean_diff)
        max_voltage = voltage_grid[max_idx]
        max_diff_val = mean_diff[max_idx]
        max_std_val = std_diff[max_idx]

        summary_data.append({
            "Sample": sample_name,
            "Mode": "Discharge",
            "Voltage (V)": round(max_voltage, 3),
            "Mean Diff Coef (cm^2/s)": max_diff_val,
            "Std Dev": max_std_val,
            "Diff Coef (cm^2/s)": f"{max_diff_val:.2e} ± {max_std_val:.2e}"
        })

#### plot formatting ####
axes[0].set_title('ICI calculated Li diff coefficients', fontsize=18, fontweight='bold')
axes[0].set_ylabel('Li Diff Coef (cm$^2$/s)', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Cell Potential (V)')
axes[0].set_ylim(-0.1e-11, 2.5e-11)
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(fontsize=12)

axes[1].set_title("Mean and Standard Deviation", fontsize=18, fontweight='bold')
axes[1].set_ylabel('Li Diff Coef (cm$^2$/s)', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Cell Potential (V)', fontsize=16, fontweight='bold')
axes[1].set_ylim(-0.1e-11, 2.5e-11)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=12)

plt.tight_layout()
fig.subplots_adjust(top=0.9)
fig.canvas.draw()

### INPUT name of study to save the figure with the correct name ###
# plt.savefig('study-1', dpi=300)
plt.show()

#### convert to dataframe ####
summary_df = pd.DataFrame(summary_data)
print("\n=== Peak Diffusion Summary ===\n")
print(summary_df[['Sample', 'Mode', 'Voltage (V)', 'Diff Coef (cm^2/s)']])

### INPUT name of study to save data summary with the correct name ###
summary_df.to_csv("diffusion_summary_study-1.csv", index=False)